# Retail Demand Forecasting Ops

End-to-end forecasting system with classical baselines plus LazyPredict, manual top-3, FLAML, and PyCaret tracks.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_prep import load_rossmann_data, merge_train_store, clean_sales_data, time_split
from src.features import engineer_calendar_features, add_lag_rolling_features, build_supervised_dataset, build_preprocessor, prepare_matrices
from src.forecast_models import (
    run_classical_baselines,
    run_lazypredict_discovery,
    select_top3_eligible_families,
    run_manual_engineering_track,
    run_flaml_track,
    run_pycaret_track,
    build_leaderboard,
    create_ops_signals,
)
from src.backtest import rolling_origin_backtest, summarize_backtest

SEED = 42
PROJECT_NAME = 'retail-demand-forecasting-ops'
PROJECT_ROOT = Path('.')
RAW_DIR = PROJECT_ROOT / 'data' / 'raw' / 'rossmann'
ARTIFACT_DIR = PROJECT_ROOT / 'artifacts'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

## 1) Business Problem and Success Criteria

Forecast store-level daily sales to improve staffing and replenishment planning.

Success criteria:
- Low sMAPE (primary), MAE (secondary), RMSE (tertiary)
- Stable performance in rolling backtests
- Actionable operational signals

## 2) Dataset Access and Data Dictionary

Dataset: Rossmann Store Sales.

Core files: `train.csv`, `store.csv`, `test.csv`.

In [ ]:
data = load_rossmann_data(RAW_DIR, sample_store_frac=0.08, random_state=SEED)
{k: v.shape for k, v in data.items()}

## 3) Data Cleaning and Leakage Checks

- Merge train and store data
- Ensure chronological ordering
- Keep only historical lag-safe features

In [ ]:
merged = merge_train_store(data['train'], data['store'])
clean = clean_sales_data(merged)
clean.head()

## 4) Feature Engineering

- Lag and rolling features
- Calendar, promo, holiday, and competition features

In [ ]:
fe = engineer_calendar_features(clean)
fe = add_lag_rolling_features(fe, target_col='Sales')
supervised = build_supervised_dataset(fe, target_col='Sales')
supervised.shape

## 5) Validation Strategy

- Strict time-aware holdout split
- Rolling-origin backtest for stability checks
- Metrics prioritized by business impact

In [ ]:
train_df, holdout_df = time_split(supervised, holdout_days=42)
train_df['Date'].min(), train_df['Date'].max(), holdout_df['Date'].min(), holdout_df['Date'].max()

In [ ]:
preprocessor = build_preprocessor(train_df.drop(columns=['Sales', 'Date']))
X_train, X_holdout, y_train, y_holdout = prepare_matrices(train_df, holdout_df, 'Sales', preprocessor)
X_train.shape, X_holdout.shape

## 6) Track 1: LazyPredict Discovery Lab

Run LazyPredict on supervised lag-feature table.

In [ ]:
lazy_table = run_lazypredict_discovery(X_train, X_holdout, y_train, y_holdout)
lazy_table.head(15)

## 7) Selection of Top 3 Eligible Models

Only eligible model families proceed into manual engineering.

In [ ]:
eligible_table, top3_families = select_top3_eligible_families(
    lazy_table, X_train, y_train, X_holdout, y_holdout, random_state=SEED
)
eligible_table, top3_families

## 8) Track 2: Manual Engineering Lab

Manual implementation of only the selected top-3 families.

In [ ]:
manual_results, manual_models, manual_preds = run_manual_engineering_track(
    top3_families, X_train, y_train, X_holdout, y_holdout, random_state=SEED
)
manual_results[['model_name', 'smape', 'mae', 'rmse']].sort_values('smape')

## 9) Track 3: FLAML Optimization Lab

Budget-aware AutoML search on lag-feature regression task.

In [ ]:
flaml_result = run_flaml_track(
    X_train=X_train,
    y_train=y_train,
    X_holdout=X_holdout,
    y_holdout=y_holdout,
    time_budget=300,
    random_state=SEED,
)
flaml_result

## 10) Track 4: PyCaret Experiment Lab

PyCaret compare/tune/finalize workflow on regression experiment.

In [ ]:
pycaret_train = train_df.drop(columns=['Date']).copy()
pycaret_holdout = holdout_df.drop(columns=['Date']).copy()
pycaret_result = run_pycaret_track(
    train_df=pycaret_train,
    holdout_df=pycaret_holdout,
    target_col='Sales',
    session_id=SEED,
    model_output_path=ARTIFACT_DIR / 'pycaret_retail_forecasting',
)
pycaret_result

## 11) Unified Leaderboard and Final Model Ranking

Compare classical baselines + manual top-3 + best FLAML + best PyCaret.

In [ ]:
classical_baselines, holdout_with_baselines = run_classical_baselines(train_df, holdout_df)
leaderboard = build_leaderboard(
    project_name=PROJECT_NAME,
    lazy_results=eligible_table,
    manual_results=manual_results,
    flaml_result=flaml_result,
    pycaret_result=pycaret_result,
    classical_baselines=classical_baselines,
)
leaderboard.head(10)

In [ ]:
leaderboard.to_csv(ARTIFACT_DIR / 'leaderboard_retail_forecasting.csv', index=False)
leaderboard[['model_name', 'library_source', 'holdout_primary_metric', 'rank_score', 'final_rank']].head(10)

## 12) Business Recommendation

After execution, explain winner rationale, safer second-best scenario, and key rejected tradeoff.

## 13) Inference / Deployment Path

Generate operations planning signals from selected winner predictions.

In [ ]:
winner = leaderboard.sort_values('final_rank').iloc[0]
if winner['library_source'] == 'manual' and winner['model_name'] in manual_preds:
    prediction = manual_preds[winner['model_name']]
elif winner['library_source'] == 'flaml':
    prediction = flaml_result.get('predictions', np.array([]))
elif winner['library_source'] == 'pycaret':
    prediction = pycaret_result.get('predictions', np.array([]))
else:
    prediction = holdout_with_baselines.get('pred_naive', pd.Series(dtype=float)).to_numpy()

ops_base = holdout_df[['Date', 'Store', 'Sales']].reset_index(drop=True).copy()
ops_base['prediction'] = prediction[:len(ops_base)] if len(prediction) else np.nan
ops_signals = create_ops_signals(ops_base, pred_col='prediction')
ops_signals.to_csv(ARTIFACT_DIR / 'ops_signals.csv', index=False)
ops_signals.head()

## 14) Monitoring / Drift / Retraining Plan

Track sMAPE drift by store cluster, holiday error spikes, and alert precision. Retrain monthly or on sustained drift.

## 15) Limitations and Next Steps

- Add richer exogenous signals (weather/events)
- Add probabilistic intervals
- Add scenario simulation for promo planning

In [ ]:
def naive_predictor(train_slice, valid_slice):
    last = train_slice.sort_values('Date').groupby('Store')['Sales'].last()
    return valid_slice['Store'].map(last).fillna(train_slice['Sales'].mean()).to_numpy()

backtest_df = rolling_origin_backtest(supervised, predictor_fn=naive_predictor, min_train_days=180, horizon_days=14, step_days=14)
backtest_summary = summarize_backtest(backtest_df)
backtest_summary.to_csv(ARTIFACT_DIR / 'backtest_summary.csv', index=False)
backtest_summary